# Providers and collections

This section introduces the main features related to providers and collections available in eodag.  
You will learn how to:
- [List available providers](#Providers-available)  
- [List available collections](#collections-available)  
- [Combine both methods](#Combine-these-two-methods)  
- [Discover collections dynamically](#collections-discovery)

Only providers that do not need authentication for search will be listed here. This notebook is executed in CI/CD without providers credentials, and EODAG hides providers needing authentication for search that do not have credentials set.

In [1]:
from eodag import EODataAccessGateway, setup_logging

setup_logging(2)

dag = EODataAccessGateway()

2026-04-09 12:05:18,988 eodag.config                     [INFO    ] Loading user configuration from: /home/julia/.config/eodag/eodag.yml
2026-04-09 12:05:19,012 eodag.config                     [WARNING ] desp_cache: skipped creating due to invalid config
2026-04-09 12:05:19,014 eodag.config                     [INFO    ] dedl: provider needing auth for search has been disabled because no credentials could be found


OperationalError: near "ORDER": syntax error

## Providers available

The attribute [EODataAccessGateway.providers](../../api_reference/core.rst#eodag.api.core.EODataAccessGateway.providers) returns pre-configured providers.

In [ ]:
dag.providers

In [3]:
dag.providers["cop_dataspace"]

Provider('cop_dataspace')

<div class="alert alert-warning">

Note

If a provider is configured to need authentication for search, and has no credentials set, it will be pruned on EODAG initialization, and will not appear in available providers list.

</div>

A [filter()](../../api_reference/provider.rst#eodag.api.provider.ProvidersDict.filter) method can be applied on this attribute to find provider(s) whose configuration matches the given free-text query parameter.

It can take a collection as an argument and will return the providers known to `eodag` that offer this collection.

In [ ]:
dag.providers.filter("S1_SAR_OCN")

Get the list of providers names from obtained [ProvidersDict](../../api_reference/provider.rst#eodag.api.provider.ProvidersDict)

In [ ]:
dag.providers.filter("S1_SAR_OCN").names

Filter providers using advanced free-text search (looks into `name`, `title`, `group` and associated collections ids)

In [17]:
dag.providers.filter("geodes or COP_DEM*")

ProvidersDict(['creodias', 'creodias_s3', 'dedl', 'earth_search', 'geodes', 'wekeo_main'])

## Collections available

The method [list_collections()](../../api_reference/core.rst#eodag.api.core.EODataAccessGateway.list_collections) returns a [CollectionsList](../../api_reference/collection.rst#eodag.api.collection.CollectionsList) instance that represents `eodag`'s internal collection catalog if used with `fetch_providers=False`. It will fetch providers for new collections and return an extended list if used with `fetch_providers=True` (default behavior).

<div class="alert alert-warning">
[Breaking change](../../breaking_changes.rst) in v4.0.0

[list_collections()](../../api_reference/core.rst#eodag.api.core.EODataAccessGateway.list_collections) returns a [CollectionsList](../../api_reference/collection.rst#eodag.api.collection.CollectionsList) instance, which is a list of [Collection](../../api_reference/collection.rst#eodag.api.collection.Collection) instances, instead of a list of dictionaries.

</div>

In [7]:
internal_catalog = dag.list_collections(fetch_providers=False)
print(f"EODAG has {len(internal_catalog)} collections stored in its internal catalog.")

EODAG has 307 collections stored in its internal catalog.


In [8]:
extended_catalog = dag.list_collections()
print(f"EODAG has {len(extended_catalog)} collections stored in its extended catalog, after having fetched providers.")

2025-11-27 15:49:28,886 eodag.config                     [INFO    ] Fetching external collections from https://cs-si.github.io/eodag/eodag/resources/ext_collections.json
2025-11-27 15:49:29,724 eodag.search.qssearch            [INFO    ] Fetching collections: https://hydroweb.next.theia-land.fr/api/v1/rs-catalog/stac/search/../collections
2025-11-27 15:49:32,626 eodag.search.qssearch            [INFO    ] Fetching collections: https://hda.data.destination-earth.eu/stac/collections


EODAG has 1449 collections stored in its extended catalog, after having fetched providers.


Note that the number of collections obtained in the extended catalog depends on the configured providers. Some require authentication, and may return a limited set of collections, depending on user permissions.

When providers are fetched for new collections, `eodag`'s collections configuration is updated in `EODataAccessGateway` instance. Extended collections list is then returned independantly of `fetch_providers` option in [list_collections()](../../api_reference/core.rst#eodag.api.core.EODataAccessGateway.list_collections):

In [9]:
called_again_catalog = dag.list_collections(fetch_providers=False)
print(f"list_collections() keeps returning {len(called_again_catalog)} collections.")

list_collections() keeps returning 1449 collections.


In [ ]:
internal_catalog

In [11]:
extended_catalog = dag.list_collections()
print(f"EODAG has {len(extended_catalog)} collections stored in its extended catalog, after having fetched providers.")

EODAG has 1449 collections stored in its extended catalog, after having fetched providers.


When providers are fetched for new collections, `eodag`'s collections configuration is updated in `EODataAccessGateway` instance. Extended collections list is then returned independantly of `fetch_providers` option in [list_collections()](../../api_reference/core.rst#eodag.api.core.EODataAccessGateway.list_collections):

In [12]:
called_again_catalog = dag.list_collections(fetch_providers=False)
print(f"list_collections() keeps returning {len(called_again_catalog)} collections.")

list_collections() keeps returning 1449 collections.


In [13]:
internal_catalog[0]

Collection(id='AERIS_IAGOS', title='In-service Aircraft for a Global Observing System', description='The mission of IAGOS is to provide high quality data throughout the tropopshere\nand lower stratosphere, and scientific expertise to understand the evolution of\natmospheric composition, air quality, and climate.\n', extent=Extent(spatial=SpatialExtent(bbox=[(-180.0, -90.0, 180.0, 90.0)]), temporal=TimeInterval(interval=[[datetime.datetime(1994, 8, 1, 0, 0, tzinfo=datetime.timezone.utc), None]])), keywords=['AERIS', 'AIRCRAFT', 'ATMOSPHERIC', 'IAGOS', 'L2'], license='other', instruments=['IAGOS-CORE', 'IAGOS-MOZAIC', 'IAGOS-CARIBIC'], processing_level='L2', eodag_sensor_type='ATMOSPHERIC')

In [14]:
internal_catalog[0].description

'The mission of IAGOS is to provide high quality data throughout the tropopshere\nand lower stratosphere, and scientific expertise to understand the evolution of\natmospheric composition, air quality, and climate.\n'

The method can take a provider name as an argument and will return the collections known to `eodag` that are offered by this provider.

In [ ]:
dag.list_collections("cop_dataspace")

As seen above, each collection resulting from [list_collections()](../../api_reference/core.rst#eodag.api.core.EODataAccessGateway.list_collections) is a [Collection](../../api_reference/collection.rst#eodag.api.collection.Collection) instance. That make [search()](./3_search.ipynb#Search-from-Collection) and [list_queryables()](./4_queryables.ipynb#Queryables-from-Collection) methods available from them.

## Combine these two methods

These two methods can be combined to find which collection is the most common in `eodag`'s catalog among all the providers.

In [16]:
availability_per_collection = []
collections_ids = [c.id for c in internal_catalog]
for collection in collections_ids:
    providers = dag.providers.filter(collection)
    availability_per_collection.append((collection, len(providers)))

availability_per_collection = sorted(availability_per_collection, key=lambda x: x[1], reverse=True)

most_common_collection, nb_providers = availability_per_collection[0]
print(f"The most common collection is '{most_common_collection}' with {nb_providers} providers offering it.")

The most common collection is 'S2_MSI_L1C' with 12 providers offering it.


These can be also used to find out which provider (as configured by `eodag`) offers the hights number of different collections.

In [18]:
availability_per_provider = []
for provider in dag.providers:
    provider_collections_ids = [
        c.id
        for c in dag.list_collections(provider, fetch_providers=False)
    ]
    availability_per_provider.append(
        (provider, len(provider_collections_ids))
    )
availability_per_provider = sorted(availability_per_provider, key=lambda x: x[1], reverse=True)
provider, nb_collections = availability_per_provider[0]
print(f"The provider with the largest number of collections is '{provider}' with {nb_collections}.")

The provider with the largest number of collections is 'dedl' with 375.


## Collections discovery

EODAG comes with a large list of pre-configured collections. Some others are available from providers catalogs but will not be configured, or are not yet configured in EODAG.

Some providers, like STAC providers, come in EODAG with a configuration describing how to discover these not-already-configured collections.

With the method [discover_collections()](../../api_reference/core.rst#eodag.api.core.EODataAccessGateway.discover_collections) 
or CLI command [eodag discover](../../cli_user_guide.rst) we can obtain a JSON configuration file that will be used as *EODAG 
external collections configuration file*.

In EODAG, the discovered *EODAG external collections configuration file* can be set to:

* a file automatically built from github actions and stored in [eodag/resources/ext_collections.json](https://cs-si.github.io/eodag/eodag/resources/ext_collections.json) (default settings)
* a custom remote or local file by setting its path in `EODAG_EXT_COLLECTIONS_CFG_FILE` environment variable (if the file is not readable, only user-modified providers will be fetched).

Then, when listing collections using [list_collections(fetch_providers=True)](../../api_reference/core.rst#eodag.api.core.EODataAccessGateway.list_collections), EODAG will first read the content of the *EODAG external collections configuration file* using [fetch_collections_list()](../../api_reference/core.rst#eodag.api.core.EODataAccessGateway.fetch_collections_list) 
then update [EODataAccessGateway](../../api_reference/core.rst#eodag.api.core.EODataAccessGateway) instance collections configuration, if needed.

The obtained collections list will contain both pre-configured and discovered collections.

![Fetch collections schema](../../_static/eodag_fetch_collections.png "Fetch collections schema")